# Experiment A-7


In [1]:
"""A-7: ANN training with Forward Propagation and Back Propagation (from scratch)."""

import numpy as np
import matplotlib.pyplot as plt


def one_hot(y: np.ndarray, n_classes: int) -> np.ndarray:
    out = np.zeros((y.size, n_classes))
    out[np.arange(y.size), y] = 1
    return out


def softmax(z: np.ndarray) -> np.ndarray:
    z_shift = z - np.max(z, axis=1, keepdims=True)
    exp_z = np.exp(z_shift)
    return exp_z / np.sum(exp_z, axis=1, keepdims=True)


class SimpleMLP:
    def __init__(self, in_dim: int, hidden_dim: int, out_dim: int, seed: int = 42):
        rng = np.random.default_rng(seed)
        self.W1 = rng.normal(0, 0.5, size=(in_dim, hidden_dim))
        self.b1 = np.zeros((1, hidden_dim))
        self.W2 = rng.normal(0, 0.5, size=(hidden_dim, out_dim))
        self.b2 = np.zeros((1, out_dim))

    @staticmethod
    def relu(x: np.ndarray) -> np.ndarray:
        return np.maximum(0, x)

    @staticmethod
    def relu_grad(x: np.ndarray) -> np.ndarray:
        return (x > 0).astype(float)

    def forward(self, X: np.ndarray):
        z1 = X @ self.W1 + self.b1
        a1 = self.relu(z1)
        z2 = a1 @ self.W2 + self.b2
        a2 = softmax(z2)
        return z1, a1, z2, a2

    def train(self, X: np.ndarray, y: np.ndarray, lr: float = 0.05, epochs: int = 2000):
        y_oh = one_hot(y, np.unique(y).size)
        history = []

        for epoch in range(1, epochs + 1):
            z1, a1, _, probs = self.forward(X)

            loss = -np.mean(np.sum(y_oh * np.log(probs + 1e-12), axis=1))
            preds = np.argmax(probs, axis=1)
            acc = (preds == y).mean() * 100
            history.append((loss, acc))

            # Backpropagation.
            m = X.shape[0]
            dz2 = (probs - y_oh) / m
            dW2 = a1.T @ dz2
            db2 = np.sum(dz2, axis=0, keepdims=True)

            da1 = dz2 @ self.W2.T
            dz1 = da1 * self.relu_grad(z1)
            dW1 = X.T @ dz1
            db1 = np.sum(dz1, axis=0, keepdims=True)

            self.W2 -= lr * dW2
            self.b2 -= lr * db2
            self.W1 -= lr * dW1
            self.b1 -= lr * db1

            if epoch % 200 == 0 or epoch == 1:
                print(f"Epoch {epoch:4d}: loss={loss:.4f}, accuracy={acc:.2f}%")

        return np.array(history)


def make_data(seed: int = 3):
    rng = np.random.default_rng(seed)
    n = 150
    c0 = rng.normal([0.0, 0.0], 0.55, size=(n, 2))
    c1 = rng.normal([2.5, 2.5], 0.55, size=(n, 2))
    c2 = rng.normal([0.0, 3.0], 0.55, size=(n, 2))
    X = np.vstack([c0, c1, c2])
    y = np.array([0] * n + [1] * n + [2] * n)
    return X, y


def main() -> None:
    X, y = make_data()
    model = SimpleMLP(in_dim=2, hidden_dim=12, out_dim=3)
    history = model.train(X, y, lr=0.05, epochs=2000)

    _, _, _, probs = model.forward(X)
    preds = np.argmax(probs, axis=1)
    final_acc = (preds == y).mean() * 100
    print(f"\nFinal training accuracy: {final_acc:.2f}%")

    epochs = np.arange(1, history.shape[0] + 1)
    plt.figure(figsize=(10, 4))
    plt.subplot(1, 2, 1)
    plt.plot(epochs, history[:, 0], color="navy")
    plt.title("Loss vs Epoch")
    plt.xlabel("Epoch")
    plt.ylabel("Cross-Entropy Loss")
    plt.grid(True, linestyle="--", alpha=0.4)

    plt.subplot(1, 2, 2)
    plt.plot(epochs, history[:, 1], color="darkgreen")
    plt.title("Accuracy vs Epoch")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy (%)")
    plt.grid(True, linestyle="--", alpha=0.4)
    plt.tight_layout()
    plt.show()


if __name__ == "__main__":
    main()



ModuleNotFoundError: No module named 'numpy'